In [19]:
import sys
import os

project_root = os.path.abspath("..")

sys.path.append(project_root)

print(project_root)

c:\Users\yhy_s\Downloads\california-housing


In [20]:
from src.Preprocessing import get_preprocessed_data

In [21]:
X_train, X_test, y_train, y_test = get_preprocessed_data(split=True)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(56775, 38)
(14223, 38)
(56775,)
(14223,)


In [22]:
print(X_train.columns)

Index(['Latitude', 'Longitude', 'PostalCode', 'StreetNumberNumeric',
       'AttachedGarageYN', 'BathroomsTotalInteger', 'BedroomsTotal',
       'FireplaceYN', 'GarageSpaces', 'LivingArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'ParkingTotal', 'PoolPrivateYN', 'Stories',
       'ViewYN', 'YearBuilt', 'LotSizeAcres', 'LotSizeArea',
       'LotSizeSquareFeet', 'AssociationFee', 'ClosePrice', 'DaysOnMarket',
       'sin_closed_date', 'Flooring_Carpet', 'Flooring_SeeRemarks',
       'Flooring_Concrete', 'Flooring_Laminate', 'Flooring_Vinyl',
       'Flooring_Wood', 'Flooring_Bamboo', 'Flooring_Brick', 'Flooring_Stone',
       'Flooring_Tile', 'Levels_MultiSplit', 'Levels_Two', 'Levels_One',
       'Levels_ThreeOrMore'],
      dtype='object')


In [23]:
X_train = X_train.drop(columns=["ClosePrice"], errors="ignore")
X_test = X_test.drop(columns=["ClosePrice"], errors="ignore")

In [24]:
print(X_train.dtypes)

Latitude                 float64
Longitude                float64
PostalCode                 int64
StreetNumberNumeric      float64
AttachedGarageYN         boolean
BathroomsTotalInteger    float64
BedroomsTotal            float64
FireplaceYN              boolean
GarageSpaces             float64
LivingArea               float64
MainLevelBedrooms        float64
NewConstructionYN        boolean
ParkingTotal             float64
PoolPrivateYN            boolean
Stories                  float64
ViewYN                   boolean
YearBuilt                float64
LotSizeAcres             float64
LotSizeArea              float64
LotSizeSquareFeet        float64
AssociationFee           float64
DaysOnMarket               int64
sin_closed_date          float64
Flooring_Carpet          float64
Flooring_SeeRemarks      float64
Flooring_Concrete        float64
Flooring_Laminate        float64
Flooring_Vinyl           float64
Flooring_Wood            float64
Flooring_Bamboo          float64
Flooring_B

In [25]:
na_counts = X_train.isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)
print(na_counts)

MainLevelBedrooms      22021
AssociationFee         16444
AttachedGarageYN        6099
ViewYN                  5686
NewConstructionYN       4529
PoolPrivateYN           3821
StreetNumberNumeric       61
FireplaceYN               38
YearBuilt                 20
ParkingTotal               1
dtype: int64


In [26]:
from sklearn.impute import SimpleImputer
import pandas as pd

imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(X_train_imputed.isna().sum().sum())
print(X_test_imputed.isna().sum().sum())

0
0


In [27]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"{name}")
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-"*30)

#### Linear Regression

In [29]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train_imputed, y_train)

pred_lr = lr.predict(X_test_imputed)

evaluate_model("Linear Regression", y_test, pred_lr)

Linear Regression
RMSE: 0.3128045321048002
R2: 0.6047270922949606
------------------------------


#### Ridge Regression

In [30]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train_imputed, y_train)

pred_ridge = ridge.predict(X_test_imputed)

evaluate_model("Ridge Regression", y_test, pred_ridge)

Ridge Regression
RMSE: 0.3128038081954678
R2: 0.6047289218171461
------------------------------


c:\Users\yhy_s\miniforge3\envs\dsc80\Lib\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.8894662679385913e-18.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


#### Random Forest

In [31]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

pred_rf = rf.predict(X_test_imputed)

evaluate_model("Random Forest", y_test, pred_rf)

Random Forest
RMSE: 0.15646855403263432
R2: 0.9010979910547237
------------------------------


##### Check feature importance

In [32]:
import pandas as pd

importances = pd.Series(
    rf.feature_importances_,
    index=X_train_imputed.columns
).sort_values(ascending=False)

importances.head(10)

Longitude                0.250054
LivingArea               0.223481
Latitude                 0.191257
PostalCode               0.175125
YearBuilt                0.032774
BathroomsTotalInteger    0.019832
StreetNumberNumeric      0.015500
Flooring_Wood            0.010606
DaysOnMarket             0.010409
LotSizeSquareFeet        0.009959
dtype: float64